# ADVIO + VGGT Colab Benchmark

This notebook prepares one ADVIO sequence, builds a baseline image set and an IMU-guided reduced image set, and then benchmarks:

- VGGT vision-only pose estimates
- IMU-guided frame pruning + VGGT pose estimates
- pose error against ADVIO ground truth
- scene-specific behavior like turns and vertical motion
- GPU-use proxies from frame-count reduction

It also includes an ARKit proxy sanity run so you can validate the pipeline before running VGGT.

## 1. Mount Drive And Point To The Repo

Upload or sync your `3d-reconstruction` repo folder into Google Drive first.
Then update `REPO_DIR` if needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/3d-reconstruction'
%cd $REPO_DIR

import os
print('Repo exists:', os.path.isdir(REPO_DIR))
print('Contents:', os.listdir(REPO_DIR)[:10])

## 2. Install Dependencies

This installs the repo requirements plus Jupyter-friendly extras.

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q notebook ipywidgets

## 3. Download One ADVIO Sequence

Sequence 15 is a calm office sequence and works well for the first sanity check.

In [ ]:
import os

ADVIO_SEQ = '15'
ADVIO_ROOT = f'data/advio-{ADVIO_SEQ}'
ADVIO_ZIP = f'data/advio-{ADVIO_SEQ}.zip'
ADVIO_URL = f'https://zenodo.org/record/1476931/files/advio-{ADVIO_SEQ}.zip'

os.makedirs('data', exist_ok=True)
if not os.path.exists(ADVIO_ZIP):
    !wget -O $ADVIO_ZIP $ADVIO_URL
if not os.path.exists(ADVIO_ROOT):
    !unzip -q -o $ADVIO_ZIP -d data

print('ADVIO root:', ADVIO_ROOT)
!find $ADVIO_ROOT -maxdepth 2 | head -n 20

## 4. Prepare Baseline And IMU-Guided Datasets

This extracts video frames and builds the IMU-guided subset.
The default settings here are the same ones we used locally for `advio-15`.

In [ ]:
PREP_ROOT = f'data/prepared/advio-{ADVIO_SEQ}'
EVERY_NTH_FRAME = 10
ROT_THRESH_DEG = 8.0
TRANS_THRESH_M = 0.0
MIN_GAP = 1

!python prepare_advio_vggt_dataset.py \
  --advio-root $ADVIO_ROOT \
  --output-root $PREP_ROOT \
  --every-nth-frame $EVERY_NTH_FRAME \
  --rotation-thresh-deg $ROT_THRESH_DEG \
  --translation-thresh-m $TRANS_THRESH_M \
  --min-gap $MIN_GAP

!cat $PREP_ROOT/summary.json

## 5. Optional Sanity Check With ADVIO ARKit Poses

This does **not** use VGGT. It exports ADVIO's ARKit poses into the benchmark format so you can verify that the pipeline, plots, and metrics all work before spending GPU time on VGGT.

In [ ]:
from pathlib import Path
import numpy as np

from src.datasets.advio import load_advio_iphone_sequence, interpolate_poses
from benchmark_advio_vggt import load_frame_metadata

seq = load_advio_iphone_sequence(ADVIO_ROOT)
base_rows = load_frame_metadata(f'{PREP_ROOT}/baseline/metadata/frame_times.csv')
guided_rows = load_frame_metadata(f'{PREP_ROOT}/imu_guided/metadata/selected_frames.csv')

base_times = np.asarray([t for _, t in base_rows], dtype=np.float64)
guided_times = np.asarray([t for _, t in guided_rows], dtype=np.float64)

base_poses = interpolate_poses(base_times, seq.arkit_timestamps, seq.arkit_poses)
guided_poses = interpolate_poses(guided_times, seq.arkit_timestamps, seq.arkit_poses)

for out_dir, rows, poses in [
    (f'outputs/advio-{ADVIO_SEQ}/arkit_proxy/baseline/poses', base_rows, base_poses),
    (f'outputs/advio-{ADVIO_SEQ}/arkit_proxy/imu_guided/poses', guided_rows, guided_poses),
]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for (name, _), pose in zip(rows, poses):
        np.savetxt(out / f'{Path(name).stem}.txt', pose, fmt='%.8f')

print('ARKit proxy poses exported.')

## 6. Run VGGT On The Two Image Sets

You have two image folders now:

- `data/prepared/advio-15/baseline/images`
- `data/prepared/advio-15/imu_guided/images`

Run your VGGT inference in whatever way you already prefer.
The only requirement for the benchmark is that you save pose files with names matching the image stems, for example:

- `frame_000000_extrinsic.txt`
- `frame_000001_extrinsic.txt`

or plain camera-to-world files like:

- `frame_000000.txt`
- `frame_000001.txt`

Set the output dirs in the next cell after your VGGT run is done.

In [ ]:
# Replace these after you run VGGT.
# For a first pipeline sanity check, leave USE_ARKIT_PROXY = True.

USE_ARKIT_PROXY = True

if USE_ARKIT_PROXY:
    BASELINE_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/arkit_proxy/baseline/poses'
    IMU_GUIDED_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/arkit_proxy/imu_guided/poses'
else:
    BASELINE_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/baseline/poses'
    IMU_GUIDED_POSE_DIR = f'outputs/advio-{ADVIO_SEQ}/imu_guided/poses'

print('Baseline pose dir:', BASELINE_POSE_DIR)
print('IMU-guided pose dir:', IMU_GUIDED_POSE_DIR)

## 7. Benchmark And Save JSON Report

In [ ]:
REPORT_JSON = f'outputs/analysis/advio-{ADVIO_SEQ}_{"arkit_proxy" if USE_ARKIT_PROXY else "vggt"}_benchmark.json'

!python benchmark_advio_vggt.py \
  --advio-root $ADVIO_ROOT \
  --baseline-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --baseline-pose-dir $BASELINE_POSE_DIR \
  --imu-guided-frames-csv $PREP_ROOT/imu_guided/metadata/selected_frames.csv \
  --imu-guided-pose-dir $IMU_GUIDED_POSE_DIR \
  --output-json $REPORT_JSON

!cat $REPORT_JSON

## 8. Generate Visualizations

In [ ]:
PLOT_DIR = f'outputs/analysis/advio-{ADVIO_SEQ}_{"arkit_proxy" if USE_ARKIT_PROXY else "vggt"}'

!python visualize_advio_benchmark.py \
  --advio-root $ADVIO_ROOT \
  --baseline-frames-csv $PREP_ROOT/baseline/metadata/frame_times.csv \
  --baseline-pose-dir $BASELINE_POSE_DIR \
  --imu-guided-frames-csv $PREP_ROOT/imu_guided/metadata/selected_frames.csv \
  --imu-guided-pose-dir $IMU_GUIDED_POSE_DIR \
  --output-dir $PLOT_DIR

In [ ]:
from IPython.display import Image, display

for name in [
    'trajectory_topdown.png',
    'gyro_keyframes.png',
    'translation_error_over_time.png',
    'frame_count_proxy.png',
]:
    path = f'{PLOT_DIR}/{name}'
    print(path)
    display(Image(filename=path))

## 9. Quick Interpretation Guide

- `ate_rmse`: lower is better for global trajectory fit
- `rotation_mean_deg` and `translation_mean`: lower is better for local frame-to-frame consistency
- `gpu_frame_proxy_ratio`: lower means fewer frames were sent to VGGT
- `gpu_attention_proxy_ratio`: lower means a potentially large transformer compute/memory reduction

A common pattern is:

- IMU-guided pruning helps compute a lot
- global error may stay similar
- local motion error can get worse if pruning is too aggressive

That tradeoff is exactly what this notebook is designed to make visible.